# 2.2 — Train a Policy on the GPU with mjlab

Trains directional myoLeg locomotion (`myoLegDirectionalForward-v0`) with thousands of parallel envs on mjlab (MuJoCo Warp + RSL-RL), then plays the policy back on the CPU env.

**Default training script:** every mjlab env is trained with the same command-line script, `scripts/train_mjlab.py <env_id>`, for example

```bash
python scripts/train_mjlab.py myoLegDirectionalForward-v0 --env.scene.num-envs 4096 --agent.max-iterations 5000
```

Always pass `--env.scene.num-envs` (1024-4096); further flags (`--agent.resume True`, `--stop-on-success`, ...) are described in the ML quickstart. Use it for real runs of any env. This notebook only adds a short demo around it (see the last line).

MyoSuite tasks have two matched halves under one `env_id`:
- **CPU** (`gym.make`, MuJoCo C++): playback, fine-tuning, debugging.
- **GPU** (mjlab): thousands of envs in parallel for fast RL.

Observation (153-d for this task) and muscle action space are identical, so a policy **trained on GPU transfers to CPU** unchanged. Parity is verified per env family (see the backend parity page): the torso-exosuit twins are *experimental* (converted observation, policy transfer not tested yet), and locomotion policies should be re-checked on the CPU env because the two backends are not step-by-step identical for them.

Training stops early by default once the success rate (`Episode_Metrics/success`, deterministic policy included) exceeds 95%; pass `--stop-on-success False` to train for the full `--agent.max-iterations`. Evaluate a checkpoint on either backend with `scripts/eval_mjlab_policy.py` (success rate, return, optional video).

**Prerequisites:** Completed notebook 1.1 · CPU parts run anywhere with `pip install -e .`; training needs Linux + CUDA and `pip install -e ".[mjlab]"`.

The demo helper, specific to this directional-leg example (CPU env check, a 5-iteration smoke training run and CPU playback of the checkpoint), is `files/2.2/directional_leg_gpu_training.py`; it is not needed for training your own envs.

In [ ]:
ENV_ID = "myoLegDirectionalForward-v0"

### 2.2.1 CPU: check the env and roll out a random policy

In [ ]:
!python files/2.2/directional_leg_gpu_training.py --cpu-demo

### 2.2.2 GPU: train

A short smoke run (needs CUDA); for a real run use the training CLI with many envs, e.g.
`python scripts/train_mjlab.py myoLegDirectionalForward-v0 --env.scene.num-envs 1024`.

In [ ]:
!python files/2.2/directional_leg_gpu_training.py --gpu-train --iterations 5

### 2.2.3 CPU: play back the trained checkpoint

In [ ]:
from pathlib import Path

import gymnasium as gym
import numpy as np

import myosuite  # noqa: F401  (registers the envs)
from myosuite.utils.checkpoint_utils import find_checkpoint, load_policy

env = gym.make(ENV_ID)
act = load_policy(env, find_checkpoint(ENV_ID, roots=(Path.cwd(), Path.cwd().parent)))  # newest mjlab run, else random

obs, _ = env.reset(seed=0)
total = 0.0
for _ in range(200):
    obs, reward, terminated, truncated, info = env.step(act(obs))
    total += float(reward)
    if terminated or truncated:
        obs, _ = env.reset()
env.close()
print(f"return over 200 steps: {total:.2f}")